In [7]:
import torch
import sys
print("Interpreter:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Interpreter: d:\maga25\NIR\.venv\Scripts\python.exe
CUDA available: False
GPU name: No GPU


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
if device.type == 'cuda':
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"Всего видеопамяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Используемое устройство: cpu


In [9]:
# Standard
import random

import numpy as np
import pandas as pd
import torch

# Third Party
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    Trainer,
    TrainingArguments,
)

# First Party
from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.util import select_by_index

In [10]:
# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

------------------------------------------------------------------------


Дообучение на датасете metr-la

In [12]:
# Путь к вашему файлу
file_path = './metr-la.h5' 

In [14]:
import h5py
import pandas as pd
import numpy as np

def load_h5_to_dataframe(file_path):
    """
    Загружает HDF5 файл (METR-LA / PEMS-BAY) в pandas DataFrame.
    Возвращает df и опционально временные метки (если есть).
    """
    with h5py.File(file_path, 'r') as f:
        print("=== Структура файла ===")
        def print_structure(name, obj):
            print(f"  {name} : {type(obj)}")
        f.visititems(print_structure)
        print("=" * 30)

        # Ищем набор данных с данными (обычно 'data' или 'raw_data' или первый 2D)
        dataset_name = None
        for key in f.keys():
            obj = f[key]
            if isinstance(obj, h5py.Dataset) and len(obj.shape) >= 2:
                dataset_name = key
                print(f"Найден датасет: '{key}' с формой {obj.shape}")
                break
        if dataset_name is None:
            # Если не нашли – возможно данные лежат в группе, ищем глубже
            for key in f.keys():
                if isinstance(f[key], h5py.Group):
                    for subkey in f[key].keys():
                        subobj = f[key][subkey]
                        if isinstance(subobj, h5py.Dataset) and len(subobj.shape) >= 2:
                            dataset_name = f"{key}/{subkey}"
                            print(f"Найден датасет: '{dataset_name}' с формой {subobj.shape}")
                            break
                    if dataset_name:
                        break
        if dataset_name is None:
            raise ValueError("Не удалось найти датасет с данными (размерность >= 2).")

        # Загружаем данные
        data = f[dataset_name][:]  # (временные шаги, количество сенсоров, ...)
        print(f"Загружены данные формы: {data.shape}")

        # Если данных больше 2D (например, (T, N, 1)), схлопываем последнюю размерность
        if data.ndim == 3 and data.shape[2] == 1:
            data = data[:, :, 0]
        elif data.ndim > 2:
            # Если несколько признаков – берём первый (или можно объединить)
            data = data[:, :, 0]
            print("Данные обрезаны до первого признака (предполагается один признак).")

        # Ищем временные метки (обычно 'time' или 'times')
        timestamps = None
        if 'time' in f:
            timestamps = f['time'][:]
        elif 'times' in f:
            timestamps = f['times'][:]
        elif 'date' in f:
            timestamps = f['date'][:]
        # Иногда метки внутри атрибутов
        if timestamps is not None:
            print(f"Найдены временные метки: {len(timestamps)} шт.")
        else:
            print("Временные метки не найдены.")

    # Создаём DataFrame
    num_sensors = data.shape[1]
    columns = [f'sensor_{i}' for i in range(num_sensors)]
    df = pd.DataFrame(data, columns=columns)

    # Если есть временные метки – добавляем их как индекс
    if timestamps is not None:
        if len(timestamps) == df.shape[0]:
            df.index = pd.to_datetime(timestamps, unit='s')  # если в секундах
        else:
            print("Количество временных меток не совпадает с числом строк.")
    else:
        # Создаём искусственный временной индекс (например, 5-минутные интервалы)
        # Для METR-LA обычно период с 2012-03-01, 5 минут
        start_time = pd.Timestamp('2012-03-01 00:00:00')
        freq = '5T'  # 5 минут
        time_index = pd.date_range(start=start_time, periods=df.shape[0], freq=freq)
        df.index = time_index
        print("Создан искусственный временной индекс (с 2012-03-01, шаг 5 мин).")

    return df

# --- Использование ---
file_path = './metr-la.h5'   # измените на ваш путь
df = load_h5_to_dataframe(file_path)
print(f"\nРазмер DataFrame: {df.shape}")
print(df.head())
print(df.info())

=== Структура файла ===
  df : <class 'h5py._hl.group.Group'>
  df/axis0 : <class 'h5py._hl.dataset.Dataset'>
  df/axis1 : <class 'h5py._hl.dataset.Dataset'>
  df/block0_items : <class 'h5py._hl.dataset.Dataset'>
  df/block0_values : <class 'h5py._hl.dataset.Dataset'>
Найден датасет: 'df/block0_values' с формой (34272, 207)
Загружены данные формы: (34272, 207)
Временные метки не найдены.


ValueError: Invalid frequency: 5T. Failed to parse with error message: ValueError("Invalid frequency: T. Failed to parse with error message: KeyError('T'). Did you mean min?")

In [ ]:
from transformers import PatchTSTConfig, PatchTSTForPrediction
import torch

# 1. Загружаем сохранённый препроцессор
tsp = TimeSeriesPreprocessor.from_pretrained("./patchtst_etth1_preprocessor")
# (если сохраняли на Drive, укажите путь)

# 2. Загружаем конфигурацию старой модели
old_config = PatchTSTConfig.from_pretrained("./patchtst_etth1_model")

# 3. Создаём новую конфигурацию для нового датасета (например, 207 каналов для METR-LA)
new_num_channels = 207   # или 325 для PEMS-BAY
new_config = PatchTSTConfig(
    num_input_channels=new_num_channels,
    context_length=old_config.context_length,   # можно оставить те же
    patch_length=old_config.patch_length,
    prediction_length=old_config.prediction_length,
    d_model=old_config.d_model,
    num_attention_heads=old_config.num_attention_heads,
    num_hidden_layers=old_config.num_hidden_layers,
    ffn_dim=old_config.ffn_dim,
    dropout=old_config.dropout,
    head_dropout=old_config.head_dropout,
    pooling_type=old_config.pooling_type,
    channel_attention=old_config.channel_attention,
    scaling=old_config.scaling,
    loss=old_config.loss,
    pre_norm=old_config.pre_norm,
    norm_type=old_config.norm_type,
    # если использовали маскировку – можно добавить do_mask_input и mask_ratio
)

# 4. Создаём новую модель
new_model = PatchTSTForPrediction(config=new_config)

# 5. Загружаем веса из старой модели (кроме входного слоя)
old_state_dict = torch.load("./patchtst_etth1_model/pytorch_model.bin")  # или используйте model.state_dict()
new_state_dict = new_model.state_dict()

# Фильтруем веса: оставляем только те, что совпадают по размеру и не относятся к input_embedding
filtered_state_dict = {}
for k, v in old_state_dict.items():
    if k in new_state_dict and 'input_embedding' not in k:
        if v.shape == new_state_dict[k].shape:
            filtered_state_dict[k] = v
            print(f"Перенесён слой: {k}")

# Обновляем веса новой модели
new_state_dict.update(filtered_state_dict)
new_model.load_state_dict(new_state_dict)

print("Модель адаптирована для нового числа каналов и загружена с предобученными весами.")